In [7]:
!pip install -q groq python-dotenv
!pip install langchain-groq

In [13]:
import getpass
import os
from groq import Groq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [14]:
BASE_MODEL = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert.
Be precise, logical, and code-focused.
Explain errors clearly and provide corrected code if necessary.
Be concise but technically rigorous."""
    },

    "billing": {
        "system_prompt": """You are a Billing Support Specialist.
Be empathetic and professional.
Explain refund policies clearly.
Guide users through next steps calmly and politely."""
    },

    "general": {
        "system_prompt": """You are a helpful general customer support assistant.
Respond conversationally and clearly."""
    }
}

In [15]:
def route_prompt(user_input):
    router_prompt = f"""
Classify this text into one of these categories:
[technical, billing, general]

Return ONLY the category name.

Text:
{user_input}
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a classification router."},
            {"role": "user", "content": router_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()
    return category

In [16]:
def process_request(user_input):

    # Step 1: Route request
    category = route_prompt(user_input)
    print(f"🔀 Routed to: {category.upper()} expert\n")

    # Safety fallback
    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    # Step 2: Generate response
    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [17]:
print(process_request("My python script is throwing an IndexError on line 5."))
print(process_request("I was charged twice for my subscription this month."))
print(process_request("What are your business hours?"))

🔀 Routed to: TECHNICAL expert

To help you resolve the issue, I'll need more information about your Python script, specifically:

1. The code (at least the relevant part)
2. The exact error message
3. The input data (if applicable)

However, I can guide you through the error.

An `IndexError` typically occurs when you're trying to access an element at an index that doesn't exist in a list, tuple, or string.

Here are some common reasons for `IndexError`:

1. Out-of-range index: You're trying to access an index that's higher than the length of the list.
2. Negative index: You're trying to access an index that's lower than 0.
3. Non-iterable: You're trying to access an index of a non-iterable object (like a non-list, non-tuple, or non-string).

Please provide your code, and I'll help you identify the issue.

Here's a basic example of how you might raise an `IndexError` in Python:

```python
# Create a list with 5 elements
my_list = [1, 2, 3, 4, 5]

# Try to access the 6th element (which 